# 📋 STR01 · 미션 03 | 단순 잔여 DOS 계산 및 1차 기준 미달 부대 식별 Baseline 알고리즘 구현

**학생용 Student · 직접 실습**  
CMS 편성: 3시간 · 교육용 가상 데이터 사용

---

## 📖 1. 상황과 이번 미션의 문제

가상의 20개 조직과 5종 물품의 기록을 이용해 **단순 잔여 DOS 계산 및 1차 기준 미달 부대 식별 Baseline 알고리즘 구현** 과제를 수행합니다.

실제 조직의 기록이 아닌 실습자료이며, 코드를 실행한 결과도 실제 현장 상태를 뜻하지 않습니다.

**이번 미션에서 해결할 문제:** 목표 DOS 대비 달성률이 50% 미만인 부대·품목을 1차 기준 미달로 식별한다.

결핍 위험 일수와 피처의 상관을 계산하고 EDA 차트 8종(분포, 추이, 템포-소모, 기상 효과, 재고 궤적, 품목별 달성률, 미달 비율 추이, 상관)을 만든다.

결과에서 군수 의사결정에 쓸 인사이트를 문장으로 정리한다.

**만들 결과물:** `notebooks/str01_eda_analysis.ipynb`, `reports/str01_eda_report.md`, `outputs/M03/results.json`, `forms/STR01_M03_forms.md`

## 📌 2. 처음 시작하는 방법

1. 아래 **환경 준비 셀**을 실행합니다. Colab에서는 메뉴의 **런타임 → 모두 실행** 대신 각 셀의 결과를 먼저 확인하면서 진행해도 됩니다.
2. 핵심 함수의 `TODO`를 AI와 함께 구현합니다. 처음에는 함수의 입력·출력과 왜 필요한지 질문하세요.
3. 첫 실행 결과를 확인하고 실패하면 오류 메시지를 AI에게 설명해 달라고 요청합니다.
4. AI가 제안한 코드 중 최소 3가지를 원자료와 실행 결과로 대조하고 수정 이유를 기록합니다.
5. 활동별 작성 칸을 채운 뒤 **실제 AI 대화 원문**을 보존하고 마지막 저장 셀로 ZIP을 만듭니다.

> **보안과 평가:** 실제 조직·위치·재고·작전 자료, 개인정보를 외부 AI에 입력하지 않습니다.

- 모델 성능은 **실제 출력 로그**로 판단하며, 예시값을 실측으로 적지 않습니다.
- 실행 성공이나 ZIP 저장은 학습 PASS·실제 승인과 다릅니다.
- 폐쇄망에서는 허용된 Python/Jupyter 환경에서 실행하세요.
- 인터넷이 필요한 `pip` 설치는 폐쇄망에서 실패할 수 있으므로 사전 패키지 준비가 필요합니다.


In [ ]:
# @title ① 실습자료 자동 준비 및 환경 설정 (ZIP 업로드 불필요) (실행 버튼을 누르세요) { display-mode: "form" }
# 이 셀은 코드 숨김 상태로 배포됩니다. 소스코드를 열지 않아도 실행할 수 있습니다.
from pathlib import Path
import os, sys, json, re, math, time, warnings, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

# 업로드 파일명은 자유이며, .zip 확장자와 유효한 ZIP 내용만 확인합니다.
current_dir = Path.cwd().resolve()
notebook_directory = globals().get('notebook_directory', current_dir)
workspace_name = 'STR01_실습프로젝트_작업공간'
WORKSPACE = next(
    (folder for folder in (*reversed(current_dir.parents), current_dir) if folder.name == workspace_name),
    current_dir / workspace_name,
)
STR01_PROJECT_READY = False

def _prepare_project_zip(raw_bytes):
    global STR01_PROJECT_READY, STR01_PROJECT_DIR
    import io
    if not zipfile.is_zipfile(io.BytesIO(raw_bytes)):
        raise ValueError('유효한 ZIP 파일이 아닙니다. 압축파일 내용을 확인하세요.')
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(raw_bytes)) as archive:
        for member in archive.infolist():
            name = member.filename.replace('\\', '/')
            target = (WORKSPACE / name).resolve()
            if not target.is_relative_to(WORKSPACE):
                raise ValueError('안전하지 않은 압축 경로가 발견되어 중단했습니다: ' + name)
            # 심볼릭 링크는 작업공간 외부 접근 가능성을 고려해 거부합니다.
            if (member.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError('심볼릭 링크가 포함된 ZIP은 허용하지 않습니다: ' + name)
        for member in archive.infolist():
            target = (WORKSPACE / member.filename.replace('\\', '/')).resolve()
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            if target.is_file():
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as source, target.open('xb') as destination:
                import shutil
                shutil.copyfileobj(source, destination)
    os.chdir(WORKSPACE)
    STR01_PROJECT_DIR = WORKSPACE
    STR01_PROJECT_READY = True
    print('✅ 실습자료 준비 완료 (기존 파일은 보존):', WORKSPACE)
    print('✅ 이제 다음 실습 코드셀을 실행할 수 있습니다.')

# 실습자료가 노트북 안에 내장되어 있어 ZIP 업로드 없이 자동으로 준비됩니다.
# 내장 파일 2개: STR01_M03_forms.md, STR01_M03_lesson.md
import base64 as _embedded_b64
_EMBEDDED_ZIP_B64 = (
    'UEsDBBQAAAAIAAAAIVze+SS4rQgAACgUAAASAAAAU1RSMDFfTTAzX2Zvcm1zLm1k5VhdTxvpFb73r3ilaKVWAtspjXaFKlWptmpXatOq27tuhUhCpXQTjDC5'
    'iOQLJ4yRwV7hJB4YyNg1G8xH6mgHMMZI0Er8lOzdvO/8hz7nnHfssaG70mp7VQUFe+b9OOc5z3nOOdxSn//5T9nb6vfZKWWaL41zpMzGS1NpqA/FutKVA1P2'
    '8aJuNjvq0z98rsITx7wIlA7W1W0THKiwH5jdotLf9HWlo3SvqKtFhe36xFG/ms3PPX40P6eMWw1PWnqvY/Y8FZ52Is9Jpcybmj7pq2h9NQyK0+ovkeuYsvfX'
    '4S5cd3X260/vKjaFPpiKHwbPo626wtXRWl99Yr4uKdPomuXnpuEos9yAA5Hr4TYVHgeR+5UyvhNtelh0gdPNFozzW6bhweDIhWfLz8NuESfAx7Jeq6uofmGO'
    'Pd2+UHfCwLdnK73umJ4fbZSNf6EPgwk875sXHbyFFTAfTgU4Ho89PMbNpuWazZoy+3x8C9bRsnQqpU+K4fElYLwwLX86xk1QJBfuZD/CRuP46uPsR3rHB9L1'
    '8N/1wcKjbuxG2YdrZKg+d8LTWuQe6kpZV3bTI+AAL/zALthS1p3+1RkZFARXZ1G1RuHFnbhd7zSGB6RSt24x3nxC5HaN00ilCvG5BXsUPsRnkU3nHp1ZBdC0'
    'YhyI132lX3TN9iEWuT5YpAo4cXJyUt3wP94Q13TPidY7eBj/kA29OkVr5FlU8rFu0qxU9buD0VcELOIXbR8A8bGTmh2mSW8X8Rk78FVZvzskCgNzMEq3W6ML'
    'YrqfO8Y/uNEmIVbyGYFqnwrJKIoj8SeM5RWiJHlV4IQrE7QvfUoX+VytAVc1WGMq8MEnxpOXoAWCgccS4e8BetTsH/8buX33M7I12v4KfgW+Dnz8Pi6avYal'
    'Xio1qSidWk5scyxFFd+0atPqw8YOHaIrp8gfpU+62KdoOb2wT81mVzerKnrj2N049O5nRD69vxP2Ad1mSXe6nDvIar81Tbe2wMXTyG1hT02/rYLKl3X9Ggbi'
    'jXEvGc83SC6PxEEApn0IBUkA6YZXRlRJS+Qtnw8p24dqvDvQ7S5thHOR+36aAgx9JAEDv4R2NnZmu26O39NdlFnaq/G2VtE028OA2rWxw9+5NrbQU/qwznnx'
    'A2mQEtOuzuRa0ygObGYDSEEQEbPTTUoQo6is2lkT45gzTqxcgWc8xo4O2HIsClDbGzSpQAwwzq5+Vxr4LLcYpxUtNwifOIaV3WhjFXlSqpq1XdKs8OwiPApG'
    'YBGmcYCZDrQfpWG/KHLWcuHxSVe3GfhTn6RQ7FWiclhmsf7vwN4E7zWEb3yQmoLUJ0obaGN2giTC7KvSx+XI6xOYJviXgLxMnA/P+rAN3oD9NdI5yLiunFsR'
    'xrn6hTfQDa7pMWyJ+FDhWjtNREDiyWRHfNbarPi6cUHV1BF4OeCUmnEi6WOHvg7ecYKOv8P//DxpQYL3Ar+NcxyuHwF21mTpQ0RM+Vp7GwSg57Mmc4sCBN8j'
    'a1UhXhq8ggM9B7nFch0GzrgU4/mAd1xIsIEFfyDhulemowtxAvxPuHTd7Ul2slseBMJ+pQzrlo1bthltQ8BKGYes1EdC6d5zTg8wC70K+dnXe/AXHQs+WuwS'
    'u0Eg06hSuTMlj5oo6mwcj2QZ/nOyDcUV3QxVSchFtNKimhkt+1IDkAWk/PwxqRj8xKydI7W/LdZ1haiOBIFfaPsqfhyHb4suhzxyTw1VXe5EBzUIgXYPqIsj'
    'ISnpFtgNJ/BbRVsuTqbnx/9UsbiISKgEbYfaG/jhcYf7S28XCjJYHNPhO6X4YS4/cz9uggvyD43jN31rL91WPWD9Q2uy1h7ykeuR6+jK1kAlyGDnCDXOGskE'
    '79WJdMl85HesM2PNpOC15ev1LeqHzL7Dzo+V74JdQk2HYHWtN2HPh5ppw8ok+h7i481tfKYJBUqFLmLa9m5cak9BOLjEvZ/+x2rMLRlc0EWSjonAQQFvHFdY'
    '6XhisWU+nkAYj/2qfr2LyyZUtFwzy6tUtZH1aIl/ghb5p/Y+mSiuDxEiygQkiTKNONt15oUlmOgG1dYDIAN40AVtH1JOUGCl/aGQJTlBvXWzZKlJrrmXPPxw'
    'abTDz6DMSVslte7qDP0Q8G86cQAFz8QMsEH24/ZfWtoV1M9i6Jl80yq/+CAzOz/7+Fn+UT6TX1rM3p6ZezibXnhGNTVpJtyXnkRMEkORnfml3IMvJ57Mzc7P'
    'PMjN558+WVh6lJtXH1ZeqXl8eYbsCdDLUezSVJEBF36ydhIT4NhieEBtFftLEn3WsjTIUo9yb/YeVTkplcSWLBkoAwv2XlJtR6gQdWo3KFx7dBimO+rN0G0P'
    'e38KgOxQ+vVR2OdEAR40n7g+XQFhwTPTOzQ7R5ZEyDxJvFgvRBq5kc+IXGKLbZyGsNNI+cb53Vw+n5un8HGqSqbSDZxMIsUglO2frgdtKg4aR/n/ZLi2UjNI'
    'KVua0TahlYpnVERBSq91RApfRvR5IINWBYfAimAQrwBcxhIEH3+RTd8hmh3SOOLCm4x5e8FEY74QJGGwRfviEX9coeM5m9YQaH+89xusKNMZnxBIJPVOjaZq'
    'xqBZHnZktnWhpV8UFr8ofFhtZ9NTiU5mgWvCkKtdlZwi7TAqAybL5FjPrX7LGQLXm54MXahxmytUyAVU/rPCShVFQRjASggYwIAaf8LIYNGR0FNq4Qoeypy3'
    '1J0ady3pkGUMjWh9eEDRH4WLOhQg3GxnpBucYjMcKS+UYtw0xGUFTjY7EtUbsuTnNkuwEK30tJrPLc3dz+W+TOjaTCx16UcLz+bvT6jFuYXc4lJyhTxJP3k4'
    'oXJPlxae4iUOzSzO5Z8+Xsqn/45EnlB/yy0+yWf4T20zeDvD37GHOC05zSOrNGJfl6S35ElRFELKJl6z09IBU+O0GSAfdM0b5Sm3maa/I0MMRjTHpwfxFXIh'
    'd0vSULFUu6jcUbMMhkAJrqH1H1BLAwQUAAAACAAAACFc7oQIjBQKAAAsGQAAEwAAAFNUUjAxX00wM19sZXNzb24ubWTtWG1PG9kV/u5fcaVopV3JsSHZtCmq'
    'tEqbaBu1SaPdfmpVgRPcii7BCJNKkfaDgQEZ7BQn8YBNxt6hMQFSVx3wgI0ErcRPSb7NvfMf+pxz79jjQtJ+7JuEBMzLvec85znPee5cEV//4quRUXFv5Lp4'
    'V6gKWdpTRUeoZlVttsXtn38tgo6lFj0hvXUxqrw9EfQ81SoI+ZeeLLWFPC7IckGoUkN2LPGjTD47PTWTFcouBx1XvmmrNzURHLXDmpVIXLki1NJCWN8POqdC'
    'vt0PnzuJROAVcFH89MFdrGUpq4EQfFk6Uc2K3HaU5ZgtaHnVbGPV865aKcu3e+ddvQZFqeoFERx6of0Mv85kx5c7p2a90N6XpaIstVLmfXpWHTlB91ToheR3'
    'q8o5xW5R9liQ3teZh7Z/ac70hE47tGtyrdpPXjUsIXfL8mULmyVFuFRRS6vKXTDxfHrn9q3PzH6qUcOaiKtIC4TVU3VY43W9v+ICxaxRe1WRHTxWrIUbq0Ie'
    'VpVjURbh+irgSySuRk+EtoWHKE61XEO+rqUOfUTZUmtHarMCNGkfvc4Al/77QHapKL6cmhfhH3rKOhDXUjdUyQk8632hSqv0tgPvhWpgSduSpdr7gj3YW686'
    'JgZFEYTSCcAotbGY3HHpzRsjnzCOu2XV6Bkoz7vhiyLeI+QucoyBMkA7XEdTaaAQbtYA4in2ZjwYQIBKPAv8Aq0XFbGGYghAD2z3wrWeuKn+uPwpKhKut5NC'
    'HVcRG2q17OD/q5oWSQ4D3Ay39sDYpKGPCI5bqCYe5piJl/0Ek32KnFjK2euvq8P5rM8MN0oD6+q6oEk8Kh2CV4tt3FGujTtCvQSyjR6uYSHEzbxu91RzJ6KQ'
    'a6PNDDKoxq80B3496EVU4rxLmXODMwRUUzRiNQ4GtvERJ8WolhrIx2D2IbCV42r6Yu8+3v5FLt8IPMesLeS6pY6dcKNIwe97yaHUQOUPorDLy7uIjh5LxVi3'
    '6GFF2T4dEzO5+ezDXO6bfDo/PzcyOp6dzIxnZjLTT/NT+dTU7NOZhyxiDMBWVdlngoBbb2Nr3WWuE3SOBAnP6zI1Ff4LV+jCc+oFZRflon/e/XFuOvOQeop6'
    'MbqoW0zI5R7V6HghuhJWLVIxt0Eo//LuA478rCpfOmNiIj/3KB0FOIg5Nft0IoVOPlLgUWhDkBhGLaJY1y2g+KY26vBPUSThK0vcusuFOa4SlLJTgBQOdbl8'
    '5hHUljcmhgKbmMzMZ9Jf3bl1+96d1OPJCSbZ3im2QiXOkIZZAphj7x9QQcEyJAEJrtpquSqtIrAcHR26A2F9Rtq2LN2d0e/hFoLXoSFWSPRNXDKZUR8OyVHJ'
    'AepCbTxH4wMpnlHjmFHjv8nNPc5TjFzLydyjfHouO5ubm89T25Pw1S2uli5aWAJZdHUbvjwsGmmkZw3C0XC6KsIyJOn0o3Whp7gieGoylx9/aHpsgtligVNj'
    'Ij+fe/RN8nE2MzP+KDeTf/J4dn4qNyPerbwQM/jnKQL31KZPbZmiJkL74GfEdIiGi4XrsIAeMIUmpe+6Zl7h2YK4n7lPKQ3G2gglRV3iedEMRB9CntB0gnqR'
    'VKKGrosUuq9aNI71G0K+PAh6q7y9qUqkK9KrSXRHn0JGJCKG0lgx/cP0pjyiqWuuQ1uKNUg984N0i+SDh11PbfOYvTdKOfw+Mz0FQmbHp7OTv83ODcg/RJGh'
    '8jH7uRmsA96Le4LltuOrbT+SMuj5tj/cIIIk2GUiRnPRp8FJK0LFg6NK3ECgqV0HfUajDfoJlnd6NMdgJoIDwr0QrpBBqbXkTo0WRJosfo7LTbW4F5wsREkQ'
    'LUO7TsIR1h25XqcrV8zfo6zWIFW49YwKRDLpItFyBSYlkTDyp0t53kVmrIaMPfZnDeOcGmeUuRlL0VztUPCEgdrylOXC80Dezdqstxu+3GgJVVsIN5iMqmUF'
    'J+UvaEX9VAwUqA5ekbvbQa9nzEkUAcmVg65A1es2z3tvXx7avKJrx+oZS/waJ26kjPMxlYJX1OHrq9p/tbhi0aACUAyzKTrefNEyS+FdIGmYDcpwmhgoW/ux'
    'VIYoMTaMGr0K82LG9fdHPmHD6FWDv1X7Dx740XBkRDmWCxyKjVxMYfywYdktYqyjkNzAwEvXgsJFabcbl2OlPbshHyiJPuLdm22NDHhiKDiAbGh+RA/G4+NR'
    'Hh/bTL9B1Fw93QhaMlFl7HJSYzuy6ZFh2CATAIJ9Qb2irJZ8u4yndDVBuHCpcR1gxS1HxBcOhw8InI4JlCpaJ6N1hBKRoCFUDAmhaXU5OJ8zOHD6hLTGUzeS'
    'cQz9JtI22pSZWppJYDSg31KXtL5PpwnLwOqQs4nBCF/23aoRWfaFcM+4mGT7ZjlmQrH2HlUIcbV2Qi1CR6SlocHoDyfBsnaJkPg4XIBAGBtvGkK9PiUT5VYS'
    'CU1OmhkYOemBPf/hCNw9RggmAonx0kJav2RmATeNV6f3ogYwvr8e5UhWSjMCMx3d9uD+l+SIaI2bbAXKe9KqgCGCTV2zOLASxr/Ro9/OfftudWckdd3sQ5DM'
    '8jnDikGAoM0pg04pTFGauMYc/UOPiJ/w7EPizRpxBzMNzFxRjbIAp6hNqeVWymrX0obWOGmaSBXtqUtHBhvtZElpsQUrnfVa7aJc9lo8HWOAyTH1kAGLaxys'
    'yGKCJczCugW+m4zJfUz8c/tKLmPCuJ3YQ/oKeSK6n3syP/sE92GW4IzyT6bn86nf5XMzdJPNkyEewEzG5hk54sM/A9rGc6SeZMb5Rd3Zff1IRmYTcAVdUqtu'
    'T76h/o/Py7g7102gs6STkjnygFooK5/hcQ/nS/z2CtwdVAXrgI7Lmu1G3WhR3SdcOh23HrexjtFuh4p5gRHgI3cX484dy+1mTlFgF+QEYnOhsomE8Y/cgoay'
    'WmAoITPXojqOpvhLih7bY/+JHxyI4XSE3KryuY7t6KBKFC+Pd7iRrX2AWpBrO2zX3QrROm6KSX+ay6S17o5xx6z0PA36uk72SR/3iERc8vMuRrZjqaYVyZnG'
    'MzYN+/Mlcc0gzkMd7vtD3p3KFo8OWcdHoY7vfaH632jf3xfIcqnjfbV9YLgDs6y9su5oVNa0NJZO67bGK8ZIDaY5ef9X1s+yeSgKVY2dtB6YtAO3pTkAbFZw'
    'ZGAH2a/VdVMrrunY/8anCqNV/QYysqfHUMw5gfpsibUvY/VNa/HtH2aMlRxU4/+T/d9hsvP4BL7NnfTA2iJFPbior8ypUJtYOhtFrnvQGp/r1tDj6V/5lpUU'
    'H3ECSfEhG5DUJiB98YsKEVn3L/sDhMkTD+VC5Gx5tRqQMp9QqThXfbal4c3GX1Zqw+TEu5HzxZ/sfOlCtIXekCcx9qBikyzbmKphs2iG8gCkROwz9Ue/Uqf1'
    'kvpjdf9bdcp8ANDHO+SrvyHIRR+zTF8kQdRn57UqqcZuIXa2+DtQSwECFAMUAAAACAAAACFc3vkkuK0IAAAoFAAAEgAAAAAAAAAAAAAApAEAAAAAU1RSMDFf'
    'TTAzX2Zvcm1zLm1kUEsBAhQDFAAAAAgAAAAhXO6ECIwUCgAALBkAABMAAAAAAAAAAAAAAKQB3QgAAFNUUjAxX00wM19sZXNzb24ubWRQSwUGAAAAAAIAAgCB'
    'AAAAIhMAAAAA'
)
_EMBEDDED_ZIP_BYTES = _embedded_b64.b64decode(_EMBEDDED_ZIP_B64)
print('📦 노트북에 내장된 실습자료를 준비합니다. ZIP 업로드는 필요하지 않습니다.')
_prepare_project_zip(_EMBEDDED_ZIP_BYTES)


## 미션 핵심 개념과 원본 실습 설명

---

## 🎯 미션 03. 단순 잔여 DOS 계산 및 1차 기준 미달 부대 식별 Baseline 알고리즘 구현

**소요시간**: 약 2.5시간

### 📋 미션 배경 및 수행 지침

#### 미션 배경 및 당위성

- 단순 잔여 DOS 계산 및 기준 미달 부대 식별 Baseline 알고리즘을 구현한다.
- 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **재고와 최근 소모량으로 단순 DOS를 계산해 기준 미달 부대를 식별하는 Baseline을 만들고, 탐색적 분석(EDA)으로 의미 있는 피처를 찾는다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.
- 이 문제를 해결하기 위해 본 미션에서는 **목표 DOS 대비 달성률이 50% 미만인 부대·품목을 1차 기준 미달로 식별 수행하고 검증하는** 실습을 체계적으로 수행합니다.
- 학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `notebooks/str01_eda_analysis.ipynb 및 EDA 요약 리포트` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)

#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 군수 의사결정에 직관적인 인사이트 제공 수준

#### 미션 PASS 조건

- 정량 기준: EDA 시각화 차트 8종 이상 생성, 유의미한 상관관계 피처 5개 이상 도출

- 진행 기준: EDA 노트북 실행 완료 및 핵심 발견점(Key Findings) 문서화

- 검증 방법: 통계적 가설 검정(p-value, 상관계수)을 통한 유의성 검증

#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.


### AI 활용 예시 · 미션 03 전용

> 나는 Python을 처음 배우는 학습자야. **단순 잔여 DOS 계산 및 1차 기준 미달 부대 식별 Baseline 알고리즘 구현**을 진행하고 있어. 다음 요구사항을 보고 핵심 함수의 입력·출력, 필요한 검증 3개를 쉬운 말로 설명해줘. `TODO` 구간만 구현할 수 있는 코드를 제안하고, 데이터를 미리 보지 않는 검증 방법과 경계 사례도 알려줘. 실행 결과는 만들지 말고 내가 실제 실행한 로그를 받은 다음 분석해줘.  
> 요구사항: src/analysis/str01_eda.py / dos_baseline: stock,mean_consumption → numpy 배열 DOS. 유한한0이상 입력을 검사하고 평균 소모0은 NaN이다. 재고0의 관찰 경보는 별도 처리하며 목표 달성률과 경보 등급을 구분한다.

**AI 답변을 받으면:** 그대로 복사하지 말고 입력 시점·자료 범위·실제 실행 여부를 직접 확인하세요. 확인되지 않은 수치는 `미측정`, 수행하지 않은 활동은 `미실시`로 남깁니다.


In [ ]:
from pathlib import Path

if not (globals().get('STR01_PROJECT_READY', False) and globals().get('STR01_PROJECT_DIR') and Path(STR01_PROJECT_DIR).is_dir()):
    print('먼저 ① 실습자료 자동 준비 및 환경 설정 셀을 실행한 뒤 이 셀을 다시 실행하세요. ZIP 업로드는 필요 없습니다.')
else:
    """재고와 최근 평균 소모량으로 잔여 보급일수(DOS)를 계산하고 목표 대비 기준 미달 부대를 식별하는 Baseline을 만들며, 8종 탐색 차트와 결핍 위험 일수와의 상관 분석으로 의미 있는 피처 5개 이상을 도출한다."""
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    # ---------------------------------------------------------------------------
    # [교육용가상자료] 가상 부대 20개 x 핵심 보급품 5종 일일 재고·소모 시뮬레이터 (실제 부대·수량·기준일수 아님)
    # 핵심성과지표(KPI) = 기준 보급일수(DOS) = 현재 재고 / 최근 평균 일일 소모량. 목표 DOS는 교육용 가정값이다.
    # 결핍 위험 일수 H = 보급이 없다고 가정할 때 현재 재고가 바닥나기까지 남은 일수(미래 실제 소모량으로 계산한 정답).
    # ---------------------------------------------------------------------------
    ITEMS = {"식량": (100, 0.3, 10), "피복": (20, 0.1, 15), "유류": (300, 1.2, 7), "탄약": (60, 1.5, 6), "수리부속": (15, 0.5, 12)}   # (기본 일소모, 템포 민감도, 목표 DOS)
    N_UNIT, DAYS = 20, 160
    UNITS = [f"가상{u:02d}부대" for u in range(1, N_UNIT + 1)]


    def make_data(seed=1, n_unit=N_UNIT, days=DAYS):
        """일 단위 재고 원장: unit, item, day, tempo(실제 작전 템포), plan(사전 계획 템포), wx(악천후), consume, resupply, stock."""
        rng = np.random.default_rng(seed)
        rows = []
        for u in range(1, n_unit + 1):
            s, ph = rng.uniform(0.7, 1.5), rng.uniform(0, 6.28)
            surge = np.zeros(days)
            for st in rng.choice(days - 10, 4, replace=False):
                surge[st:st + int(rng.integers(5, 10))] = rng.uniform(0.4, 0.8)
            tempo = 1 + 0.25 * np.sin(2 * np.pi * np.arange(days) / 30 + ph) + surge
            plan = tempo + rng.normal(0, 0.08, days)
            wx = (rng.random(days) < 0.15).astype(int)
            for it, (b, sens, tg) in ITEMS.items():
                stock, off = tg * b * s * rng.uniform(0.8, 1.2), int(rng.integers(0, 4))
                for t in range(days):
                    c = b * s * (1 + sens * (tempo[t] - 1)) * (1 + 0.12 * wx[t]) * rng.normal(1, 0.08)
                    r = max(tg * b * s - stock, 0) * rng.uniform(0.9, 1.1) * (0.5 if rng.random() < 0.08 else 1) if (t % 4 == off and t > 0) else 0.0   # 4일 주기 정기 보급: 목표 재고까지 채움(가끔 수송 차질로 절반)
                    stock = max(stock + r - c, 0.0)
                    rows.append((f"가상{u:02d}부대", it, t, tempo[t], plan[t], wx[t], c, r, stock))
        return pd.DataFrame(rows, columns=["unit", "item", "day", "tempo", "plan", "wx", "consume", "resupply", "stock"])


    def make_samples(d, lo=21, hi=125):
        """(부대, 품목, 기준일 t0)별 특성(피처)과 정답 H. 피처는 t0 시점까지 알 수 있는 값과 사전 계획·기상 예보만 사용한다."""
        out = []
        for (u, it), g in d.groupby(["unit", "item"]):
            c, stk, rs = g["consume"].values, g["stock"].values, g["resupply"].values
            pl, wx, tp = g["plan"].values, g["wx"].values, g["tempo"].values
            cs = np.cumsum(c)
            tg = ITEMS[it][2] * ITEMS[it][0]
            for t0 in range(lo, hi):
                idx = np.searchsorted(cs, cs[t0] + stk[t0])
                if idx >= len(cs) or idx <= t0:
                    continue
                c3, c7, c14 = c[t0 - 2:t0 + 1].mean(), c[t0 - 6:t0 + 1].mean(), c[t0 - 13:t0 + 1].mean()
                last = [k for k in range(t0, t0 - 15, -1) if rs[k] > 0]
                out.append({"unit": u, "item": it, "t0": t0, "H": idx - t0, "stock": stk[t0], "c3": c3, "c7": c7, "c14": c14,
                            "dos7": stk[t0] / c7, "dos3": stk[t0] / c3, "trend": c7 / c14, "burst": c3 / c7,
                            "plan7": pl[t0 + 1:t0 + 8].mean(), "plan_ratio": pl[t0 + 1:t0 + 8].mean() / tp[t0], "tempo0": tp[t0],
                            "wx_fc": wx[t0 + 1:t0 + 8].mean(), "since_rs": (t0 - last[0]) if last else 15,
                            "rs_avg": rs[t0 - 13:t0 + 1].sum() / 14, "fill": stk[t0] / tg, "item_code": list(ITEMS).index(it)})
        return pd.DataFrame(out)


    def mape(a, b):
        # [TODO : 학생 실습 3] 단순 잔여 DOS 계산 및 1차 기준 미달 부대 식별 Baseline 알고리즘 구현 관련 핵심 로직을 구현하세요.
        # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
        # <<< 여기에 코드를 작성하세요 >>>
        pass  # 작성 후 pass를 제거하세요.
    d = make_data()
    d["c7"] = d.groupby(["unit", "item"])["consume"].transform(lambda s: s.rolling(7, min_periods=3).mean())
    d["dos"] = d["stock"] / d["c7"]
    d["target"] = d["item"].map(lambda i: ITEMS[i][2])
    d["rate"] = d["dos"] / d["target"]

    # 1) Baseline: 기준 미달 = 달성률 50% 미만(주의), 25% 미만(심각 후보)
    T = 100
    now = d[d["day"] == T]
    low = now[now["rate"] < 0.5].sort_values("rate")
    print(f"[Baseline - {T}일째] 단순 DOS = 재고 / 최근 7일 평균 소모량 [교육용가상자료]")
    print(f"부대 x 품목 {len(now)}건 중 달성률 50% 미만 {len(low)}건({len(low) / len(now):.0%}), 25% 미만 {(now['rate'] < 0.25).sum()}건")
    print("기준 미달 상위 5건:")
    print(low.head(5)[["unit", "item", "stock", "c7", "dos", "target", "rate"]].round(2).to_string(index=False))
    unit_cnt = low.groupby("unit").size().sort_values(ascending=False)
    print("미달 품목이 많은 부대 3곳:", unit_cnt.head(3).to_dict())

    # 2) EDA용 표본(결핍 위험 일수 H 포함)과 상관 분석
    s = make_samples(d[["unit", "item", "day", "tempo", "plan", "wx", "consume", "resupply", "stock"]])
    feat = [c for c in s.columns if c not in ("unit", "item", "t0", "H")]
    corr = s[feat + ["H"]].corr()["H"].drop("H").sort_values(key=abs, ascending=False)
    sig = corr[corr.abs() >= 0.3]
    print(f"\n[결핍 위험 일수 H와의 상관] 표본 {len(s)}건, |r| 0.3 이상 피처 {len(sig)}개")
    print(corr.round(3).head(8).to_string())

    # 3) EDA 차트 8종
    fig, ax = plt.subplots(2, 4, figsize=(17, 7))
    ax[0, 0].hist(now["dos"].clip(upper=25), bins=20, color="tab:blue"); ax[0, 0].set_title("1 DOS distribution")
    dm = d.groupby("day")["consume"].mean(); ax[0, 1].plot(dm.index, dm.values); ax[0, 1].set_title("2 Mean daily consumption")
    sub = d[d["item"] == "탄약"].sample(600, random_state=1); ax[0, 2].scatter(sub["tempo"], sub["consume"], s=6); ax[0, 2].set_title("3 Tempo vs consumption")
    ax[0, 3].boxplot([d[d["wx"] == 0]["consume"] / d["c7"].mean(), d[d["wx"] == 1]["consume"] / d["c7"].mean()]); ax[0, 3].set_xticklabels(["fair", "bad wx"]); ax[0, 3].set_title("4 Weather effect")
    for u in UNITS[:3]:
        g = d[(d["unit"] == u) & (d["item"] == "유류")]; ax[1, 0].plot(g["day"], g["stock"], label=u[2:4])
    ax[1, 0].legend(fontsize=7); ax[1, 0].set_title("5 Stock trajectories")
    ax[1, 1].boxplot([d[d["item"] == i]["rate"].dropna().clip(upper=3) for i in ITEMS]); ax[1, 1].set_xticklabels([f"i{k + 1}" for k in range(5)]); ax[1, 1].set_title("6 Achievement rate by item")
    cnt = d[d["day"] >= 14].groupby("day")["rate"].apply(lambda x: (x < 0.5).mean()); ax[1, 2].plot(cnt.index, cnt.values, color="tab:red"); ax[1, 2].set_title("7 Share below 50% target")
    ax[1, 3].barh(sig.index[::-1][:8], sig.values[::-1][:8], color="tab:green"); ax[1, 3].set_title("8 Correlation with H")
    plt.tight_layout(); plt.show()
    n_chart = ax.size
    r_t = float(d[d["item"] == "탄약"][["tempo", "consume"]].corr().iloc[0, 1])
    wx_up = float(d[d["wx"] == 1]["consume"].mean() / d[d["wx"] == 0]["consume"].mean() - 1)
    srs = s.groupby(pd.cut(s["since_rs"], [-1, 0, 1, 2, 3, 20]))["dos7"].mean()
    print(f"\nEDA 시각화 차트 {n_chart}종 생성")
    print(f"인사이트: (1) 탄약 소모와 작전 템포의 상관 r={r_t:.2f}. (2) 악천후일 평균 소모가 {wx_up:+.1%}. (3) 보급 후 경과일별 평균 DOS: {[round(float(v), 1) for v in srs.values]} (보급 직전이 가장 낮음)")
    print("의미 있는 상관 피처:", ", ".join(sig.index[:6]))
    print(f"목표(EDA 차트 8종 이상, 유의미한 상관 피처 5개 이상): 차트 {n_chart}종 {'달성' if n_chart >= 8 else '미달'}, 피처 {len(sig)}개 {'달성' if len(sig) >= 5 else '미달'}")


## 📋 3. 학습활동 4단계

### 활동 1: 무엇을 확인하고 계획하나요?

**이번 활동의 과제:**

#### 1. 단계 개요

- **목표:**
  - 원본 목표, 기본 코드, 그리고 제공된 학습 자료를 면밀히 분석하여, 단순 잔여 DOS 계산 시 미달 기준(예: 50% → 70% 등)의 변화가 부대별 우선순위 판단에 미치는 영향 을 사전에 예측하고 판단합니다.
  - 아울러 탐색적 데이터 분석(EDA)과 Baseline 구축 과정에서 AI 에이전트에 위임할 분석 범위, 데이터 제약 사항, 그리고 검증 방법을 체계적으로 수립합니다.
- **수행 방식:**
  - 막연한 코드 작성이나 AI의 답변에 의존하지 않고, 군수 의사결정의 정확성을 높이기 위한 명확한 사전 규칙과 검증 계획을 문서화합니다.

#### 2. 세부 수행 절차

- **원본 목표 및 자료 분석:**
  - 재고와 최근 소모량으로 단순 DOS를 계산해 기준 미달 부대를 식별하는 Baseline을 만들고, 탐색적 분석(EDA)으로 유의미한 피처를 찾는 데 필요한 입력 데이터, 기간, 단위, 그리고 사용 가능한 시점의 제약 조건을 파악합니다.
- **미달 기준 변화에 따른 우선순위 영향 검토:**
  - 미달 판단 기준(예: DOS / 목표 DOS < 0.5 등)이 상향되거나 조정될 때, 지원 우선순위가 필요한 부대 리스트와 작전 리스크 평가가 어떻게 달라지는지 사전에 분석하고 정의합니다.
- **dos_baseline 입출력 계약 및 AI 위임 범위 정의:**
  - 향후 구현할 핵심 함수인 dos_baseline 의 입출력 계약 조건을 이해하고, AI 에이전트에 맡길 EDA 차트 생성 및 코드 구현 범위를 설정하여 검증 계획을 문서( forms/STR01_M03_forms.md )에 명시합니다.

#### 3. 주의 및 검증 사항

- 입력 시점이나 기간, 분모가 서로 다른 데이터를 동일한 조건으로 잘못 해석하지 않도록 데이터 해석의 엄정성을 유지해야 합니다.
- 본 단계에서 수립된 사전 판단과 검증 계획은 다음 단계(AI와 코드 구현 및 첫 성공 실행)…

#### 완료 확인:

- 이 활동의 확인 근거 M03 계획: 재고와 최근 소모량으로 단순 DOS를 계산해 기준 미달 부대를 식별하는 Baseline을 만들고, 탐색적 분석(EDA)으로 의미 있는 피처를 찾는다.에 필요한 입력·기간·단위·사용 가능 시점과 dos_baseline의입출력 계약을 설명하고 본인의사전판단·AI위임·검증계획을 기록했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다.
- 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

#### 내가 기록할 것:

- 직접 확인한 입력·원본 조건:
- AI에 실제로 요청한 질문과 받은 답변의 위치:
- 실제 실행 결과/오류 메시지:
- 잘못된 설명, 수정한 코드, 수정 이유:

### 활동 2: AI와 코드 초안을 만들어 실행하기

**이번 활동의 과제:**

#### 1. 단계 개요

- **목표:**
  - 핵심 함수인 dos_baseline 을 구현·저장하고 첫 성공 실행 결과를 온전히 보존합니다.
  - 입력값으로 stock (재고)과 mean_consumption (평균 소모량)을 받아 넘파이(numpy) 배열 형태의 기준 보급일수(DOS)를 산출하며, 유한한 0 이상 입력을 검사하고 평균 소모량이 0일 때는 NaN 으로 처리합니다.
  - 아울러 재고가 0인 상태의 관찰 경보는 별도로 분리하여 처리하고, 목표 달성률과 경보 등급을 엄격히 구분합니다.
- **수행 방식:**
  - AI와 협업하여 요구되는 데이터 검사 규칙과 Baseline 산출 로직을 실제 코드에 반영하고, 변경 과제(미달 기준을 50%에서 70%로 변경)를 수행하여 미달 건수와 순위 변화를 비교 분석합니다.

#### 2. 세부 수행 절차

- 핵심 함수 dos_baseline 구현 및 첫 성공 실행: stock 과 mean_consumption 을 입력받아 numpy 배열 DOS를 반환하는 함수를 구현합니다.
- 유한한 0 이상 입력 여부를 검사하고, 평균 소모량이 0일 경우 산출 불가 처리를 위해 NaN 으로 변환하며, 재고가 0인 관찰 경보와 목표 달성률 및 경보 등급을 명확히 구분하여 첫 성공 실행 로그를 보존합니다.
- 변경 과제 수행 (미달 기준 50% -> 70% 상향 조정): 미달 판단 기준을 기존 50%에서 70%로 변경하여 코드를 수정하고 재실행합니다.
- 기준 변경에 따라 부대별 미달 건수와 지원 우선순위 순위가 어떻게 변동되는지 비교 검증합니다.
- **차트 8종 분석 및 실제 출력 보존:**
  - 탐색적 데이터 분석(EDA) 과정에서 생성되는 차트 8종 각각에 대해 질문, 관찰, 판단 내용을 기록하고, 실제 실행 코드와 출력 결과물을 누락 없이 저장합니다.

#### 3. 주의 및 검증 사항

- 미달 기준 변경(50% -> 70%) 시 단순 비율 조정에 …

#### 완료 확인:

- 이 활동의 확인 근거 M03 구현: src/analysis/str01_eda.py의 dos_baseline를저장하고계약〔stock,mean_consumption → numpy 배열 DOS. 유한한0이상 입력을 검사하고 평균 소모0은 NaN이다.
- 재고0의 관찰 경보는 별도 처리하며 목표 달성률과 경보 등급을 구분한다.〕을충족하는 첫성공실행·로그·표/그림을보존했는가?
- 이후Lesson의추가변경을실제코드에반영했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다.
- 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

#### 내가 기록할 것:

- 직접 확인한 입력·원본 조건:
- AI에 실제로 요청한 질문과 받은 답변의 위치:
- 실제 실행 결과/오류 메시지:
- 잘못된 설명, 수정한 코드, 수정 이유:

### 활동 3: 원자료와 결과를 대조·수정하기

**이번 활동의 과제:**

#### 1. 단계 개요

- **목표:**
  - 앞서 구현 및 수정된 Baseline 코드와 탐색적 데이터 분석(EDA) 스크립트를 저장하고 재실행합니다.
  - EDA 시각화 차트 8종 이상(또는 하나의 PNG 내 8개 패널)을 생성하여 결핍 위험 일수와 유의미한 상관관계를 가진 피처를 5개 이상 도출하고, 군수 의사결정에 직관적인 인사이트를 제공할 수 있도록 정교화합니다.
  - 아울러 AI의 설명 및 코드 선택 3건 이상을 실제 군수 도메인 자료 및 실행 결과와 철저히 대조하고 보완된 답변을 검증합니다.
- **수행 방식:**
  - 단편적인 차트 생성에 그치지 않고, 상관계수와 유의수준(ppp-value), 미달 기준(DOS/목표 DOS<0.5)의 엄격한 적용 조건을 실제 데이터와 대조하며 비판적으로 교차 검증합니다.

#### 2. 세부 수행 절차

- **수정 코드 저장 및 EDA 차트 8종 재실행:**
  - 수정된 분석 코드를 안전하게 저장하고 재실행하여, 결핍 위험 일수와 연관된 8종 이상의 시각화 차트와 5개 이상의 유의미한 피처 상관분석 결과를 도출합니다.
- **통계적 지표 대조 및 미달 기준 엄격 적용:**
  - 표본 수와 상관계수를 정확히 기록하되, 상관관계가 곧 인과관계를 의미하지 않음을 유의하고, 미래 보급 없음 조건의 소진일수(HHH)는 정답 값이므로 피처 목록에서 제외합니다.
  - 미달 기준은 명확히 DOS/목표 DOS<0.5 조건으로 판정하며, 일반 정상/주의 경보 기준과 혼동하지 않도록 기준을 엄격히 대조합니다.
  - AI 설명·코드 선택 3건 이상 근거 대조 및 보완: AI가 제안하거나 설명한 코드 선택 및 분석 방향 중 핵심적인 3건 이상을 선정하여, 실제 군수 실무 자료와 실행 결과 데이터에 비추어 타당한지 교차 검증하고 보완된 답변을 기록합니다.

#### 3. 주의 및 검증 사항

- 미달 기준은 명확하게 지정된 수식(DOS/목표 DOS<0.5)을 준수해야 하며, 단순…

#### 완료 확인:

- **판정에 필요한 근거**
  - 이 활동의 확인 근거 M03 검증: [필수] Baseline DOS·EDA — EDA 시각화 차트 8종 이상을 생성하고 결핍 위험 일수와 유의미한 상관이 있는 피처를 5개 이상 도출했으며, 인사이트가 군수 의사결정에 직관적인가.
  - 실제기간·분모·조건·수치로판정하고 최초/수정실행을비교했는가?
- **미션별 확인 사항**
  - 미달은 DOS/목표 DOS<0.5이며 정상/주의 경보와 같은 기준이 아닙니다.
  - EDA 8종은 한 PNG 안의 8개 패널일 수 있습니다.
  - 원본의 |r|≥0.3 기준과 p값을 구분해 표본 수·상관 계수를 기록합니다.
  - H는 미래 보급 없음 조건의 소진일수로 정의된 정답이며 피처 목록에 넣지 않습니다.
  - 상관은 인과가 아닙니다.
- **AI 검토 기록**
  - AI 주장/선택3건을근거로검토·보완·재실행했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다. 실…

#### 내가 기록할 것:

- 직접 확인한 입력·원본 조건:
- AI에 실제로 요청한 질문과 받은 답변의 위치:
- 실제 실행 결과/오류 메시지:
- 잘못된 설명, 수정한 코드, 수정 이유:

### 활동 4: 최종 결과와 수행 상태 기록하기

**이번 활동의 과제:**

#### 1. 단계 개요

- **목표:**
  - 미션 3(단순 잔여 DOS 계산 및 1차 기준 미달 부대 식별 Baseline 알고리즘 구현) 전 과정을 수행하며 산출된 탐색적 데이터 분석 노트북 , EDA 종합 분석 보고서 , 결과 데이터 파일 , 그리고 실습 기록 서식 을 누락 없이 집대성합니다.
  - 아울러 정량적 목표 달성 여부, 미측정 영역, 문서 작성 상태 및 교관(또는 시스템) 승인 대기 상태를 명확히 구분하여 최종 판단을 기록합니다.
- **수행 방식:**
  - 단편적인 파일 제출에 그치지 않고, 이번 미션에서 도출한 Baseline 산출 결과와 EDA 시각화 차트 8종 및 주요 피처 상관분석 인사이트를 객관적으로 검수하여 후속 미션(미션 4: 작전 템포 변동 및 기상 악화 연계 부대별 결핍 위험 일자 정밀 예측 고도화)으로 온전히 인계할 수 있도록 패키지를 정비합니다.

#### 2. 세부 수행 절차

- **핵심 산출 파일 패키지 확정 및 제출:**
  - EDA 분석 노트북인 notebooks/str01_eda_analysis.ipynb , EDA 결과 보고서인 reports/str01_eda_report.md , 처리 결과 데이터인 outputs/M03/results.json , 그리고 실습 기록 서식인 forms/STR01_M03_forms.md 의 실제 작성 파일들을 최종 점검하고 제출 상태로 정리합니다.
- **Baseline 코드 실행 결과 및 AI 대화 기록 연계 보존:**
  - dos_baseline 함수 구현 및 미달 기준 변경(50% -> 70%)에 따른 전후 비교 결과와, 분석 과정에서 AI 에이전트와 나눈 핵심 질의응답 및 코드 선택 근거 대화 내용을 실습 대화 보존 위치에 명확히 연결합니다.
- **상태별 최종 판단 기록 작성:**
  - 정량적 목표 달성 여부, 현장 여건상 미처 측정하지 못한 영역, 문서 작성 완료 상태, 그리고 실제 교관 승인 대기…

#### 완료 확인:

- 이 활동의 확인 근거 M03 완료: notebooks/str01_eda_analysis.ipynb, reports/str01_eda_report.md, outputs/M03/results.json, forms/STR01_M03_forms.md의실제작성·최종값·대화보존위치·선택이유가연결되는가?
- 미달·미측정·문서미작성·실제승인대기를정확히표시했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다.
- 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

#### 내가 기록할 것:

- 직접 확인한 입력·원본 조건:
- AI에 실제로 요청한 질문과 받은 답변의 위치:
- 실제 실행 결과/오류 메시지:
- 잘못된 설명, 수정한 코드, 수정 이유:


## 🔍 4. 평가 기준과 확인할 자료

- [정량적 평가] [필수] Baseline DOS·EDA — EDA 시각화 차트 8종 이상을 생성하고 결핍 위험 일수와 유의미한 상관이 있는 피처를 5개 이상 도출했으며, 인사이트가 군수 의사결정에 직관적인가. PASS 증거 제공된조건·단위·기간·분모와실제수치/건수/항목근거를연결합니다. 목표미달·0기준·미측정은그대로표시합니다. 보완 방법 기준별빠진항목과미달원인을찾아수정하고재검증합니다.
- **[정성적 평가] AI 활용: 본인 계획·실제 대화·코드 변경·검토3건·보완·재실행·전후 비교·최종 선택.**
  - **PASS 증거**
    - **수행 조건과 판정 기준**
      - 미달은 DOS/목표 DOS<0.5이며 정상/주의 경보와 같은 기준이 아닙니다.
      - EDA 8종은 한 PNG 안의 8개 패널일 수 있습니다.
      - 원본의 |r|≥0.3 기준과 p값을 구분해 표본 수·상관 계수를 기록합니다.
      - H는 미래 보급 없음 조건의 소진일수로 정의된 정답이며 피처 목록에 넣지 않습니다.
      - 상관은 인과가 아닙니다.
    - **보완 방법**
      - 보완 방법 근거없는원인·규정·완료추정은수정하거나확인필요로보류합니다.
- **[진행 추적 및 완료 기준] 정성·검증:**
  - **수행 조건과 판정 기준**
    - 미달은 DOS/목표 DOS<0.5이며 정상/주의 경보와 같은 기준이 아닙니다.
    - EDA 8종은 한 PNG 안의 8개 패널일 수 있습니다.
    - 원본의 |r|≥0.3 기준과 p값을 구분해 표본 수·상관 계수를 기록합니다.
    - H는 미래 보급 없음 조건의 소진일수로 정의된 정답이며 피처 목록에 넣지 않습니다.
    - 상관은 인과가 아닙니다.
  - **제출 파일 및 완료 기록**
    - 파일: notebooks/str01_eda_analysis.ipynb, reports/str01_eda_report.md, outputs/M03/results.json, forms/STR01_M03_forms.md 실제AI요청·답변·근거대조3건이상·보완·재확인·본인최종선택과작성/실행기록이연결됩니다.
    - 최종ZIP생성은실제학습완료·교관승인 인증이아닙니다.
  - **보완 방법**
    - 보완 방법 미작성·미실행·검토누락을보완하고교관검토상태를별도로표시합니다.
**실제 출력 없이 달성으로 표시하지 않습니다.**

In [ ]:
# @title ⑤ 개별 산출물·학습기록 통합 입력, 저장 및 ZIP 다운로드 (실행 버튼을 누르세요) { display-mode: "form" }
# 입력 폼이 나타나면 소스코드 대신 버튼을 이용하세요. Colab/Jupyter에서 ipywidgets 필요.
from pathlib import Path

if not (globals().get('STR01_PROJECT_READY', False) and globals().get('STR01_PROJECT_DIR') and Path(STR01_PROJECT_DIR).is_dir()):
    print('먼저 ① 실습자료 자동 준비 및 환경 설정 셀을 실행한 뒤 이 셀을 다시 실행하세요. ZIP 업로드는 필요 없습니다.')
else:
    def _load_notebook_widgets():
        from importlib import import_module, invalidate_caches
        import subprocess
        import sys

        try:
            return import_module('ipywidgets')
        except ImportError:
            print('이 노트북 커널에 ipywidgets가 없어 설치합니다.')

        try:
            subprocess.run(
                [sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
                 '--retries', '0', '--timeout', '10', 'ipywidgets'],
                check=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=60,
            )
            invalidate_caches()
            return import_module('ipywidgets')
        except (subprocess.SubprocessError, OSError, ImportError):
            print('이 커널에서 ipywidgets를 설치하거나 불러오지 못했습니다.')
            print('설치 가능한 환경에서는 새 셀에서 %pip install ipywidgets 를 실행한 뒤 이 셀을 다시 실행하세요.')
            print('설치할 수 없는 환경에서는 제공된 Markdown 작성 양식을 사용하세요.')
            return None

    widgets = _load_notebook_widgets()
    if widgets is not None:
        import json, csv, io, ast, zipfile, datetime, pathlib
        from pathlib import Path
        import os
        from urllib.parse import quote
        from IPython.display import HTML, display, Markdown, clear_output

        MISSION_CODE = 3
        DELIVERABLE_PATHS = ['notebooks/str01_eda_analysis.ipynb', 'reports/str01_eda_report.md', 'outputs/M03/results.json', 'forms/STR01_M03_forms.md']
        PROJECT_NAME = 'STR01'
        WORK_ROOT = Path(STR01_PROJECT_DIR).resolve()

        RECORD_DIR = WORK_ROOT / f'STR01_M{MISSION_CODE:02d}_학습기록'
        RECORD_DIR.mkdir(parents=True, exist_ok=True)
        ZIP_FILENAME = f'STR01_m{MISSION_CODE:02d}_review.zip'

        # 반드시 CMS 지정 경로만 사용하며 상대 경로의 작업공간 이탈을 방지합니다.
        def safe_path(relative):
            target = (WORK_ROOT / relative).resolve()
            if not target.is_relative_to(WORK_ROOT):
                raise ValueError('작업공간 밖의 경로는 사용할 수 없습니다.')
            return target

        def valid_text(path, content):
            suffix = Path(path).suffix.lower()
            if suffix == '.json':
                json.loads(content)
            elif suffix == '.ipynb':
                doc = json.loads(content)
                if not isinstance(doc, dict) or not isinstance(doc.get('cells'), list) or 'nbformat' not in doc:
                    raise ValueError('노트북 JSON에 cells 또는 nbformat 정보가 없습니다.')
            elif suffix == '.py':
                ast.parse(content)
            elif suffix == '.csv':
                if not content.strip():
                    raise ValueError('CSV 데이터가 비어 있습니다.')
                rows = list(csv.reader(io.StringIO(content)))
                if not rows or not rows[0] or not any(x.strip() for x in rows[0]):
                    raise ValueError('CSV 첫 행에 열 이름을 입력하세요.')
                if any(len(row) != len(rows[0]) for row in rows if row):
                    raise ValueError('CSV 열 개수가 행마다 다릅니다.')

        # 산출물 입력 화면 — 파일별 별도 탭
        editors = {}
        file_tabs = []
        file_buttons = {}
        for relative in DELIVERABLE_PATHS:
            suffix = Path(relative).suffix.lower()
            hint = {
                '.md': '보고서·양식의 제목, 분석 결과, 근거와 수정 이유를 Markdown으로 작성합니다.',
                '.csv': '첫 줄에 열 이름, 다음 줄부터 데이터 행을 쉼표로 구분하여 입력합니다.',
                '.json': '키와 값이 있는 유효한 JSON 형식으로 실제 결과를 입력합니다.',
                '.py': '직접 구현하고 확인한 Python 소스코드를 입력합니다.',
                '.ipynb': '노트북 JSON 원문을 입력하거나 기존 파일에서 불러옵니다. JSON 노트북 형식을 유지하세요.'
            }.get(suffix, '실제 작성한 내용을 입력합니다.')
            editor = widgets.Textarea(value='', placeholder='여기에 내용을 작성하세요. 입력하지 않은 파일은 저장하지 않습니다.',
                                      layout=widgets.Layout(width='100%', height='250px'))
            status = widgets.HTML('<span style="color:#777">아직 저장하지 않았습니다.</span>')
            load_btn = widgets.Button(description='기존 파일 불러오기', icon='folder-open', button_style='')
            save_btn = widgets.Button(description='이 파일 저장', icon='save', button_style='success')
            editors[relative] = {'editor': editor, 'status': status, 'loaded': False, 'saved': False, 'dirty': False, 'saved_value': ''}
            def on_edit(change, key=relative):
                state = editors[key]
                if state['loaded']:
                    state['dirty'] = True
                    state['status'].value = '<span style="color:#a65">● 수정됨 — 저장 버튼을 누르세요.</span>'
            editor.observe(on_edit, names='value')
            def load_clicked(button, key=relative):
                state = editors[key]
                target = safe_path(key)
                if not target.is_file():
                    state['status'].value = '<span style="color:#a22">파일이 아직 없습니다. 직접 입력 후 저장하세요.</span>'
                    return
                if state['dirty']:
                    state['status'].value = '<span style="color:#a22">미저장 내용이 있습니다. 불러오기 전에 저장하세요.</span>'
                    return
                try:
                    state['loaded'] = False
                    state['saved_value'] = target.read_text(encoding='utf-8')
                    state['editor'].value = state['saved_value']
                    state['loaded'] = True
                    state['dirty'] = False
                    state['saved'] = True
                    state['status'].value = '<span style="color:#286">파일을 불러왔습니다. 변경하면 다시 저장하세요.</span>'
                except Exception as exc:
                    state['status'].value = '<span style="color:#a22">불러오기 오류: '+str(exc)+'</span>'
            def save_clicked(button, key=relative):
                save_one(key)
            load_btn.on_click(load_clicked)
            save_btn.on_click(save_clicked)
            file_buttons[relative] = {'load':load_btn,'save':save_btn}
            file_tabs.append(widgets.VBox([
                widgets.HTML('<b>저장 경로:</b> <code>'+relative+'</code><br><span>'+hint+'</span><br><span style="color:#a55">기존 파일이 있다면 저장 시 덮어씁니다. 실습 완료·승인과는 다릅니다.</span>'),
                editor, widgets.HBox([load_btn, save_btn]), status
            ]))

        def save_one(relative):
            state = editors[relative]
            value = state['editor'].value
            if not value.strip():
                state['status'].value = '<span style="color:#a22">비어 있어 저장하지 않았습니다.</span>'
                return False
            try:
                valid_text(relative, value)
                target = safe_path(relative)
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_text(value, encoding='utf-8')
                state['saved_value'] = value
                state['loaded'] = True
                state['dirty'] = False
                state['saved'] = True
                state['status'].value = '<span style="color:green">✓ 저장 완료: '+relative+'</span>'
                return True
            except Exception as exc:
                state['status'].value = '<span style="color:#a22">저장 실패: '+str(exc)+'</span>'
                return False

        # 학습기록 및 AI 원문, 자료 대조, 검토 3건 이상
        record_specs = [
         ('plan','학습 계획·수행 기록','이번 미션에서 직접 수행한 내용, 실행 조건 및 최초 결과를 작성하세요.'),
         ('ai_request','실제 AI 질문 원문','실제로 입력한 질문 원문만 작성하세요. 사용하지 않았다면 비워 둡니다.'),
         ('ai_response','실제 받은 AI 답변 원문','실제로 받은 답변 전체를 붙여 넣으세요. 임의로 생성한 대화는 기록하지 않습니다.'),
         ('evidence','실제 출력·자료 대조 근거','실제 출력값, 원자료 위치, 전후 비교 및 계산 근거를 적으세요.'),
         ('revision','검토 및 수정 내용 종합','검토 3건의 발견사항과 코드를 수정·재실행한 기록을 종합하세요.'),
         ('decision','최종 판단과 미달·미측정·미실시 사유','달성 여부와 남은 부분, 근거 및 추가 확인사항을 적으세요.'),
        ]
        record_widgets = {}
        record_items = []
        for key, title, hint in record_specs:
            field = widgets.Textarea(placeholder=hint, layout=widgets.Layout(width='100%',height='115px'))
            record_widgets[key] = field
            record_items.append(widgets.VBox([widgets.HTML('<b>'+title+'</b><br><small>'+hint+'</small>'),field]))

        reviews = []
        for i in range(3):
            claim = widgets.Textarea(placeholder=f'{i+1}번째 AI 주장·코드 선택 또는 점검 대상',layout=widgets.Layout(width='100%',height='65px'))
            ground = widgets.Textarea(placeholder='원자료·실행 출력의 위치 및 비교한 근거',layout=widgets.Layout(width='100%',height='65px'))
            action = widgets.Textarea(placeholder='맞음·수정·보류, 수정 내용 및 재확인 결과',layout=widgets.Layout(width='100%',height='65px'))
            reviews.append((claim,ground,action))
            record_items.append(widgets.VBox([widgets.HTML(f'<b>검토 {i+1} (최소 3건 중 {i+1}번째)</b>'),claim,ground,action]))

        status_check = widgets.SelectMultiple(options=['달성','미달','미측정','미실시','추가 확인 필요'],
                                              value=(),description='수행 상태',layout=widgets.Layout(width='70%',height='115px'))
        approval = widgets.Dropdown(options=['미확인 / 교관 승인 대기','실제 교관 검토 완료 (기록 확인 필요)'],
                                    value='미확인 / 교관 승인 대기', description='교관 상태',layout=widgets.Layout(width='85%'))
        record_notice = widgets.HTML('<span style="color:#666">입력 후 [학습기록 저장]을 누르세요. 미작성·미측정은 그대로 표시합니다.</span>')
        record_dirty = {'value': False}
        def _mark_record_dirty(change):
            record_dirty['value'] = True
            record_notice.value = '<span style="color:#a65">● 학습기록에 저장하지 않은 변경이 있습니다.</span>'
        for field in list(record_widgets.values()) + [v for group in reviews for v in group] + [status_check,approval]:
            field.observe(_mark_record_dirty,names='value')

        # 앞서 작성한 학습기록을 자동으로 불러와 이어서 수정할 수 있습니다.
        previous_audit = RECORD_DIR/'learner_audit.json'
        if previous_audit.is_file():
            try:
                previous = json.loads(previous_audit.read_text(encoding='utf-8'))
                for key, field in record_widgets.items():
                    field.value = str(previous.get(key,'') or '')
                for group, past in zip(reviews, previous.get('reviews',[])):
                    for widget, key in zip(group,('claim','evidence','revision_and_recheck')):
                        widget.value = str(past.get(key,'') or '')
                status_check.value = tuple(x for x in previous.get('status',[]) if x in status_check.options)
                if previous.get('instructor_approval') in approval.options:
                    approval.value = previous['instructor_approval']
                record_dirty['value'] = False
                record_notice.value = '<span style="color:#286">이전 학습기록을 불러왔습니다. 수정 후 다시 저장하세요.</span>'
            except Exception as exc:
                record_notice.value = '<span style="color:#a22">기존 학습기록 읽기 오류: '+str(exc)+'</span>'

        def audit_object():
            return {
                'project': PROJECT_NAME, 'mission': MISSION_CODE,
                'plan':record_widgets['plan'].value, 'ai_request':record_widgets['ai_request'].value,
                'ai_response':record_widgets['ai_response'].value, 'evidence':record_widgets['evidence'].value,
                'revision':record_widgets['revision'].value, 'decision':record_widgets['decision'].value,
                'reviews':[{'claim':a.value,'evidence':b.value,'revision_and_recheck':c.value} for a,b,c in reviews],
                'status':list(status_check.value),'instructor_approval':approval.value,
                'saved_at':datetime.datetime.now().isoformat(timespec='seconds')
            }

        def save_audit(button=None):
            try:
                record = audit_object()
                (RECORD_DIR/'learner_audit.json').write_text(json.dumps(record,ensure_ascii=False,indent=2),encoding='utf-8')
                (RECORD_DIR/'AI_대화원문.txt').write_text('실제 AI 질문 원문\n'+record['ai_request']+'\n\n실제 AI 답변 원문\n'+record['ai_response'],encoding='utf-8')
                sections = [f'# STR01 미션 {MISSION_CODE:02d} 학습기록']
                for name,key in [('학습 계획·수행 기록','plan'),('실제 출력·자료 대조 근거','evidence'),('검토 및 수정 내용','revision'),('최종 판단','decision')]:
                    sections.append('## '+name+'\n'+(record[key] or '(미작성)'))
                for i,item in enumerate(record['reviews'],1):
                    sections.append(f"## 검토 {i}\n- 검토 대상: {item['claim'] or '(미작성)'}\n- 대조 근거: {item['evidence'] or '(미작성)'}\n- 수정 및 재확인: {item['revision_and_recheck'] or '(미작성)'}")
                sections.append('## 상태\n'+(', '.join(record['status']) or '미선택')+'\n\n교관 상태: '+record['instructor_approval'])
                (RECORD_DIR/'학습기록.md').write_text('\n\n'.join(sections)+'\n',encoding='utf-8')
                record_dirty['value'] = False
                completed = sum(all(v.strip() for v in [a.value,b.value,c.value]) for a,b,c in reviews)
                record_notice.value = f'<span style="color:green">✓ 학습기록 저장 완료. 근거가 모두 입력된 검토: {completed}/3건. 저장은 PASS나 교관 승인과 다릅니다.</span>'
                return True
            except Exception as exc:
                record_notice.value = '<span style="color:#a22">학습기록 저장 실패: '+str(exc)+'</span>'
                return False

        save_audit_btn = widgets.Button(description='학습기록 저장',icon='save',button_style='success')
        save_audit_btn.on_click(save_audit)
        record_tab = widgets.VBox(record_items+[widgets.HTML('<b>최종 수행 상태</b> — 해당되는 항목을 선택하세요. 선택만으로 자동 평가되지 않습니다.'),status_check,approval,save_audit_btn,record_notice])

        tabs = widgets.Tab(children=file_tabs+[record_tab])
        for i, path in enumerate(DELIVERABLE_PATHS):
            tabs.set_title(i, f'산출물 {i+1}')
        tabs.set_title(len(file_tabs),'학습기록')

        summary_output = widgets.Output()
        all_btn = widgets.Button(description='작성한 산출물 전체 저장', icon='save',button_style='info')
        zip_btn = widgets.Button(description='저장 파일 ZIP 생성',icon='archive',button_style='warning')
        dl_btn = widgets.Button(description='ZIP 다운로드',icon='download',button_style='')

        def save_all(button):
            ok = 0
            for rel in DELIVERABLE_PATHS:
                if editors[rel]['editor'].value.strip() and save_one(rel):
                    ok += 1
            with summary_output:
                summary_output.clear_output()
                print(f'산출물 {ok}개 저장 완료. 파일별 상태를 확인하세요. 학습기록은 별도 [학습기록 저장] 버튼을 누르세요.')

        def build_zip(button):
            with summary_output:
                summary_output.clear_output()
                if record_dirty['value']:
                    print('❌ 학습기록에 미저장 변경 내용이 있습니다. [학습기록 저장]을 먼저 누르세요.')
                    return
                dirty = [p for p,s in editors.items() if s['dirty']]
                if dirty:
                    print('❌ 미저장 변경 내용이 있습니다. 아래 파일들을 먼저 저장하세요:')
                    for p in dirty: print(' -',p)
                    return
                # 기존에 제공된 파일은 ZIP에 임의로 포함하지 않고 현재 화면에서 불러오거나 저장한 파일만 수집합니다.
                saved = [p for p,s in editors.items() if s['saved'] and safe_path(p).is_file()]
                audit_files = [p for p in RECORD_DIR.rglob('*') if p.is_file()]
                if not saved and not audit_files:
                    print('❌ 저장된 결과물이 없습니다. 산출물 또는 학습기록을 저장하세요.')
                    return
                dest = WORK_ROOT / ZIP_FILENAME
                with zipfile.ZipFile(dest,'w',compression=zipfile.ZIP_DEFLATED) as archive:
                    for rel in saved: archive.write(safe_path(rel), arcname=rel)
                    for f in audit_files: archive.write(f, arcname=str(f.relative_to(WORK_ROOT)))
                print('✅ ZIP 생성:', dest)
                print('산출물:',len(saved),'개 / 학습기록:',len(audit_files),'개')
                print('※ ZIP 생성은 학습 완료 또는 교관 승인을 의미하지 않습니다.')

        def download_zip(button):
            with summary_output:
                summary_output.clear_output()
                target = WORK_ROOT / ZIP_FILENAME
                if not target.is_file():
                    print('먼저 [저장 파일 ZIP 생성]을 눌러 주세요.')
                    return
                try:
                    from google.colab import files as colab_files
                    colab_files.download(str(target))
                    print('다운로드를 요청했습니다.')
                except ImportError:
                    start_dir = globals().get('notebook_directory', WORK_ROOT.parent)
                    href = quote(Path(os.path.relpath(target, start_dir)).as_posix())
                    display(HTML(f'<a href="{href}" download>ZIP 다운로드: {target.name}</a>'))

        all_btn.on_click(save_all)
        zip_btn.on_click(build_zip)
        dl_btn.on_click(download_zip)

        print(f'✅ STR01 미션 {MISSION_CODE:02d} 산출물·학습기록 입력 화면')
        display(Markdown('**사용 순서:** ① 각 산출물 탭에서 작성하거나 기존 파일 불러오기 → ② 파일별 저장 → ③ 학습기록 탭에서 실제 AI 원문·근거·검토 3건·최종 상태 입력 및 저장 → ④ ZIP 생성 → ⑤ 다운로드.'))
        display(Markdown('**주의:** 원본 실습의 TODO는 직접 실행해야 합니다. 입력창은 실제 수행 내용을 기록하기 위한 기능이며, 미작성 결과를 자동으로 완성하지 않습니다.'))
        display(tabs,widgets.HBox([all_btn,zip_btn,dl_btn]),summary_output)
